# Lesson 1 — Score Reading & Preprocessing

In this lesson you will write **6 functions** that turn a score file into data your
program can work with — how to split the **left hand, right hand, and both hands**,
how to turn pitch names into strings that can be **spoken out loud**, and two
different ways of playing back a chord.

---

## Background: where this score came from

The original material is a **scanned image** of a piano score. Optical Music
Recognition (OMR) tools turn that image into MusicXML.

Typical sources, ranked by quality on this kind of scan:

- **MuseScore** ([musescore.com](https://musescore.com)) — usually the best
  result, but it is a paid source.
- **Audiveris** ([github.com/Audiveris/audiveris](https://github.com/Audiveris/audiveris)) —
  open source and free. This notebook uses its output.
- **oemer** ([github.com/BreezeWhite/oemer](https://github.com/BreezeWhite/oemer)) —
  also open source and free, but weaker here (the same scan came out as 28
  measures instead of 24).

**MuseScore > Audiveris > oemer.** Audiveris and oemer cost nothing; MuseScore
does not.

**OMR makes mistakes.** This score is missing the title and the tempo marking.
The final task asks you to **listen with your ears** — just knowing that errors
exist is enough; you do not need to log them in this lesson.

---

## How to use this notebook

- Every task has a `TODO` cell — replace `raise NotImplementedError` with your implementation
- Every task is followed by a **check** cell — if it runs without an error, your implementation is correct

In [ ]:
!apt-get update -qq
!apt-get install -y fluidsynth fluid-soundfont-gm musescore3 xvfb libfontconfig1 > /dev/null
!pip install -q music21

import os, glob, subprocess, tempfile, copy
import music21 as m21
from IPython.display import Audio, display
from music21 import environment
from google.colab import drive

os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-dir"
os.makedirs("/tmp/runtime-dir", exist_ok=True)

us = environment.UserSettings()
us["musescoreDirectPNGPath"] = "/usr/bin/mscore3"
us["musicxmlPath"] = "/usr/bin/mscore3"

drive.mount("/content/drive")

data_path = "/content/drive/MyDrive/coding/Musicproject/"
print(os.listdir(data_path))

SCORE_PATH = os.path.join(data_path, "allegretto_audiveris.mxl")
print("music21", m21.__version__)
print("score:", SCORE_PATH)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['lesson1_score_parsing_BLANK_EN.json', 'allegretto_audiveris.mxl', 'lesson1_score_parsing_BLANK_EN.ipynb', 'Copy of lesson1_score_parsing_BLANK_EN 2.ipynb']
music21 9.9.2
score: /content/drive/MyDrive/coding/Musicproject/allegretto_audiveris.mxl


In [ ]:
_sf2 = (glob.glob("/usr/share/sounds/sf2/*.sf2")
        + glob.glob("/usr/share/soundfonts/*.sf2"))
SF2 = _sf2[0] if _sf2 else None

def listen(stream, tempo=108):
    """Render a music21 stream as piano audio and play it inline in the notebook."""
    if SF2 is None:
        raise RuntimeError("No soundfont found — re-run the environment setup cell above")
    s = copy.deepcopy(stream)

    for el in list(s.recurse().getElementsByClass(m21.bar.Repeat)):
        el.activeSite.remove(el)
    for m in s.getElementsByClass(m21.stream.Measure):
        if isinstance(m.leftBarline, m21.bar.Repeat):
            m.leftBarline = None
        if isinstance(m.rightBarline, m21.bar.Repeat):
            m.rightBarline = None
    s.insert(0, m21.tempo.MetronomeMark(number=tempo))
    mid = tempfile.mktemp(suffix=".mid")
    wav = tempfile.mktemp(suffix=".wav")
    s.write("midi", fp=mid)
    # Options must come BEFORE the two file arguments —
    # fluidsynth 2.6 rejects options that follow a filename
    subprocess.run(["fluidsynth", "-ni", "-F", wav, "-r", "44100", SF2, mid],
                   check=True, capture_output=True)
    display(Audio(wav))

---

## 💡 Warm-up: how to read the music21 docs

music21 models a score as boxes inside boxes: `Score` → `Part` (right hand / left hand) → `Measure` → `Note` / `Chord`.
All of these are subclasses of [`Stream`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html), so they behave the same way.

A concrete doc example — [`music21.note.Note`](https://music21.org/music21docs/moduleReference/moduleNote.html#note):

| What you want | Where to look | Code |
|---|---|---|
| Build a middle C | Constructor | `n = m21.note.Note("C4")` |
| Pitch name `"C"` | Attributes → `.name` | `n.name` |
| MIDI number `60` | `.pitch` → [`.midi`](https://music21.org/music21docs/moduleReference/modulePitch.html#music21.pitch.Pitch.midi) | `n.pitch.midi` |
| Octave `4` | `.octave` | `n.octave` |
| Duration in beats `1.0` | `.quarterLength` | `n.quarterLength` |

Run the cell below and see for yourself:

In [ ]:
# ===== Warm-up example: create a note and inspect its attributes =====
# 1. Create a C4 quarter note
sample_note = m21.note.Note("C4", quarterLength=1.0)

# 2. The common attributes documented in the official docs:
print("name:", sample_note.name)                  # "C"
print("octave:", sample_note.octave)              # 4
print("MIDI number:", sample_note.pitch.midi)     # 60
print("quarterLength:", sample_note.quarterLength)  # 1.0

# 3. Listen to the note!
sample_stream = m21.stream.Stream()
sample_stream.append(sample_note)
listen(sample_stream)

name: C
octave: 4
MIDI number: 60
quarterLength: 1.0


---

# Task 1: `load_score`

Read a MusicXML file into a [`Score`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html#score) object. A warm-up.

| Doc | Description | Code |
|---|---|---|
| [`m21.converter.parse(path)`](https://music21.org/music21docs/moduleReference/moduleConverter.html#music21.converter.parse) | Pass a path; the format is detected automatically | `return m21.converter.parse(path)` |

In [ ]:
def load_score(path):
    """Read a MusicXML file and return a music21 Score object."""
    # TODO: use music21's converter to read the file and return it
    return m21.converter.parse(path)

In [ ]:
# ===== Check 1 =====
score = load_score(SCORE_PATH)
assert isinstance(score, m21.stream.Score), "the returned object is not a Score"
assert len(score.parts) == 2, f"expected 2 parts, got {len(score.parts)}"
print("✅ Task 1 passed")
print("   parts:", [(p.id, type(p).__name__) for p in score.parts])

✅ Task 1 passed
   parts: [('P1-Staff1', 'PartStaff'), ('P1-Staff2', 'PartStaff')]


---

# Task 2: `get_hand`

Pull the right hand, left hand, or both hands out of a `Score`.

When music21 reads `<part id="P1"><staves>2</staves>`, it automatically splits it into two `PartStaff` objects:

```
score.parts[0]   id='P1-Staff1'   → right hand
score.parts[1]   id='P1-Staff2'   → left hand
```

## ⚠️ Two traps

**You cannot use the name.** Both `PartStaff` objects have `partName == 'Piano'`.

**You cannot use the clef.** In this piece treble clef happens to mean right hand, but that is luck — in real piano scores the left hand switches to treble clef when it plays high, and you would grab the wrong hand. **The staff number is the only reliable signal**, and it is at the end of `.id`.

## Spec

| `hand` | Returns |
|---|---|
| `"right"` | the part whose id ends with `'-Staff1'` |
| `"left"` | the part whose id ends with `'-Staff2'` |
| `"both"` | **the whole `score`** (the two hands must stay time-aligned) |
| anything else | `raise ValueError` |

If no matching id is found, fall back to `score.parts[0]` / `score.parts[-1]`.

| Step | Doc | Code |
|---|---|---|
| Iterate parts | [`score.parts`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html#score) | `for p in score.parts:` |
| Test the id | [`part.id`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html#part) | `if str(p.id).endswith("-Staff1"): return p` |

In [ ]:
HANDS = ("right", "left", "both")


def get_hand(score, hand="right"):
    """Return the requested hand. 'both' returns the whole score so the hands stay time-aligned."""
    # TODO: raise ValueError if hand is not in HANDS

    # TODO: hand == "both" -> return the score directly

    # TODO: decide whether to look for '-Staff1' or '-Staff2' based on hand,
    #       then walk score.parts and find the one whose id ends with it

    # TODO: fallback if nothing matches — right hand returns score.parts[0],
    #       left hand returns score.parts[-1]
    if hand not in HANDS:
        raise ValueError("Error")

    if hand == "both":
        return score

    suffix = "-Staff1" if hand == "right" else "-Staff2"
    for part in score.parts:
        if str(part.id).endswith(suffix):
            return part

    return score.parts[0] if hand == "right" else score.parts[-1]

In [ ]:
# ===== Check 2 =====
rh = get_hand(score, "right")
lh = get_hand(score, "left")

assert str(rh.id).endswith("-Staff1"), f"got {rh.id}, which is not the right hand"
assert str(lh.id).endswith("-Staff2"), f"got {lh.id}, which is not the left hand"
assert get_hand(score, "both") is score, "'both' should return the whole score"
assert get_hand(score) is rh, "the default should be the right hand"

try:
    get_hand(score, "third")
    raise AssertionError("an unknown hand should raise ValueError")
except ValueError:
    pass

def count(part, cls):
    return len(list(part.recurse().getElementsByClass(cls)))

rh_notes, rh_chords = count(rh, m21.note.Note), count(rh, m21.chord.Chord)
lh_notes, lh_chords = count(lh, m21.note.Note), count(lh, m21.chord.Chord)
assert (rh_notes, rh_chords) == (52, 0), f"right hand should be 52 single notes and 0 chords, got {rh_notes} and {rh_chords}"
assert (lh_notes, lh_chords) == (20, 28), f"left hand should be 20 single notes and 28 chords, got {lh_notes} and {lh_chords}"

print("✅ Task 2 passed")
print(f"   right hand {rh.id}: {rh_notes} single notes, {rh_chords} chords")
print(f"   left hand  {lh.id}: {lh_notes} single notes, {lh_chords} chords")
print("   Every chord in this piece is in the left hand — that fact will drive the playback modes later")

✅ Task 2 passed
   right hand P1-Staff1: 52 single notes, 0 chords
   left hand  P1-Staff2: 20 single notes, 28 chords
   Every chord in this piece is in the left hand — that fact will drive the playback modes later


---

# Task 3: `score_profile`

Read a piece's basic facts and use them as its defaults — **do not hard-code 24 measures or 2/4 time**, or you will have to edit the code for every new piece.

## The dict to return

| key | Contents |
|---|---|
| `n_measures` | number of measures (int) |
| `time_signature` | time signature string such as `'2/4'`; `None` if absent |
| `key_fifths` | number of sharps/flats in the key signature; **`None` if there is no key signature** |
| `rh_midi_low` / `rh_midi_high` | MIDI numbers of the lowest / highest right-hand note |
| `rh_n_chords` / `lh_n_chords` | how many chords each hand has |
| `default_tempo` | hard-coded `108` |

## ⚠️ This score has no key signature

Audiveris did not detect one at all (the source file has no `<key>` tag), so music21 returns `None`. **Your code must handle that without crashing** — OMR output is frequently missing pieces.

| What you want | Doc | Code |
|---|---|---|
| Measure count | [`Measure`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html#measure) | `len(rh.getElementsByClass(m21.stream.Measure))` |
| Time signature | [`TimeSignature.ratioString`](https://music21.org/music21docs/moduleReference/moduleMeterBase.html#music21.meter.base.TimeSignature.ratioString) | `ts = score.recurse().getElementsByClass(m21.meter.TimeSignature).first()`<br>`ts.ratioString if ts else None` |
| Key signature | [`KeySignature.sharps`](https://music21.org/music21docs/moduleReference/moduleKey.html#music21.key.KeySignature.sharps) | `ks = score.recurse().getElementsByClass(m21.key.KeySignature).first()`<br>`ks.sharps if ks else None` |
| Range | [`.pitch.midi`](https://music21.org/music21docs/moduleReference/modulePitch.html#music21.pitch.Pitch.midi) | `midis = [n.pitch.midi for n in rh.recurse().getElementsByClass(m21.note.Note)]` |

> The chord counts look useless right now, but **the playback mode in Task 6 exists precisely because of those two numbers**. Measure first, design second.

In [ ]:
# def score_profile(score):
#     """Analyze the score and return a dict of basic facts.

#     ⚠️ This score has no key signature — key_fifths must return None without crashing.
#     """
#     # TODO: use get_hand to grab the right and left hands
#     right = get_hand(score, "right")
#     left = get_hand(score, "left")
#     # TODO: n_measures — the right hand's measure count
#     right_measure = len(right.GetElementsByClass('Measure'))
#     # TODO: time_signature — None if not found
#     time_signature = right.getElementsByClass('TimeSignature')
#     time_signature = time_signature.ratioString if time_signature is not None else None
#     # TODO: key_fifths — None if not found (which is the case for this score)

#     # TODO: rh_midi_low / rh_midi_high — pitch.midi of every right-hand note

#     # TODO: rh_n_chords / lh_n_chords — the Chord count for each hand

#     # TODO: default_tempo is hard-coded to 108; return the whole dict
#     raise NotImplementedError

In [ ]:
# # ===== Check 3 =====
# prof = score_profile(score)
# expected = {
#     "n_measures": 24,
#     "time_signature": "2/4",
#     "key_fifths": None,      # <- this score has no key signature
#     "rh_midi_low": 62,       # D4
#     "rh_midi_high": 77,      # F5
#     "rh_n_chords": 0,        # <- the right hand has no chords at all
#     "lh_n_chords": 28,       # <- every chord is in the left hand
#     "default_tempo": 108,
# }
# for k, v in expected.items():
#     assert k in prof, f"missing key: {k}"
#     assert prof[k] == v, f"{k} should be {v!r}, got {prof[k]!r}"
# print("✅ Task 3 passed")
# for k, v in prof.items():
#     print(f"   {k:16} {v}")

---

# Task 4: `slice_measures`

Extract a range of measures — this is what "play measures 1 through 2" actually does.

## Spec

- `slice_measures(part, 1, 2)` → measures 1 and 2 (**inclusive on both ends**)
- Range outside the score → `raise ValueError`, and the message must **say how many measures there are**

The error message will later be read aloud to the user, so write it in plain language ("this piece only has 24 measures").

## ⚠️ `"both"` is a `Score`, not a `Part`

`get_hand(score, "both")` returns the whole `Score`, and **a `Score` has no `Measure` objects of its own** — the measures live inside its two parts. Counting directly gives you **0**, and every two-hand playback blows up.

So: count yourself first, and only look inside the parts if that comes up empty — which is exactly what `count_measures` does (already written for you).

| Feature | Doc | Code |
|---|---|---|
| Total measures | [`Measure`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html#measure) | `total = len(part.getElementsByClass(m21.stream.Measure))` |
| Range check | — | `if start < 1 or end > total or start > end:`<br>`    raise ValueError(f"invalid range, this piece has {total} measures")` |
| Slice measures | [`Stream.measures(start, end)`](https://music21.org/music21docs/moduleReference/moduleStreamBase.html#music21.stream.base.Stream.measures) | `return part.measures(start, end)` |

In [ ]:
def count_measures(stream):
    """Measure count. If given a Part, count directly; if given a whole Score, count the longest part."""
    own = stream.getElementsByClass(m21.stream.Measure)
    if len(own):
        return len(own)
    parts = list(stream.parts) if hasattr(stream, "parts") else []
    return max((len(p.getElementsByClass(m21.stream.Measure)) for p in parts), default=0)


def slice_measures(stream, start, end):
    """Extract measures start through end (inclusive on both ends).

    Raise ValueError on an invalid range; the message must include the total measure count.
    """
    total = count_measures(stream)
    # TODO: validate start / end and raise ValueError if invalid
    #       check that: start is at least 1, end does not exceed total, start is not greater than end
    if start < 1 or end > total or start > end:
        raise ValueError(f"invalid range, this piece has {total} measures")

    # TODO: return the sliced measures
    return stream.measures(start, end)

In [ ]:
# ===== Check 4 =====
seg = slice_measures(rh, 1, 2)
nums = [m.number for m in seg.getElementsByClass(m21.stream.Measure)]
assert nums == [1, 2], f"expected measures 1 and 2, got {nums}"

names = [n.name for n in seg.recurse().getElementsByClass(m21.note.Note)]
assert names == ["C", "C", "C", "B"], f"the first two right-hand measures should be C C C B, got {names}"

lengths = [float(n.quarterLength)
           for n in seg.recurse().getElementsByClass(m21.note.Note)]
assert lengths == [1.0, 1.0, 1.0, 1.0], f"all four should be quarter notes, got {lengths}"

# An out-of-range request must raise ValueError, and the message must mention 24
try:
    slice_measures(rh, 1, 50)
    raise AssertionError("measure 50 is out of range, this should raise ValueError")
except ValueError as e:
    assert "24" in str(e), f"the error message should tell the user the total measure count, got: {e}"

try:
    slice_measures(rh, 5, 2)
    raise AssertionError("start greater than end should raise ValueError")
except ValueError:
    pass

print("✅ Task 4 passed")
print("   right hand, first two measures:", names)

✅ Task 4 passed
   right hand, first two measures: ['C', 'C', 'C', 'B']


---

# Task 5: `to_note_names`

Turn a passage into a list of speakable English pitch names — the core of `read_notes` mode.

## Spec

Take a stream, return a `list[str]` in performance order.

| Input | Output |
|---|---|
| single note `C` | `'C'` |
| sharp `C#` | `'C sharp'` |
| flat `B-` | `'B flat'` |
| chord `[C3, E3, G3]` | `'C E G'` (low to high, space-separated) |
| rest | **skip it** |

## ⚠️ Three details

**Write `'C sharp'`, not `'C#'`** — this string goes straight to speech synthesis. How a TTS engine pronounces `C#` is unpredictable. **The output format is dictated by the fact that it has to be spoken.**

**In music21 a flat is `-`, not `b`** — B-flat has `.name == 'B-'`.

**Use `flatten()`, not `recurse()`** — `flatten()` sorts by **time**; `recurse()` gives you the whole right hand and then the whole left hand, which scrambles the order for two-hand playback.

In [ ]:
def to_note_names(stream):
    """Turn a stream into a list of speakable English pitch names.

    'C#' -> 'C sharp', 'B-' -> 'B flat', chord -> 'C E G', rests are skipped.
    """
    def spoken(name):
        """Turn a music21 pitch name into something speakable. 'C#' -> 'C sharp'"""
        # TODO: replace '#' with ' sharp' and '-' with ' flat'
        return name.replace("#", " sharp").replace('-', ' Flat')

    names = []
    # TODO: walk stream.flatten().notesAndRests in order
    #       chord -> convert each pitch's name and join them with spaces
    #       single note -> convert it directly
    #       rest -> skip
    for e in stream.flatten().notesAndRests:
      if isinstance(e, m21.chord.Chord):
          names.append(" ".join(spoken(p.name) for p in e.pitches))
      elif isinstance(e, m21.note.Note):
          names.append(spoken(e.name))
    return names

In [ ]:
# ===== Check 5 =====
assert to_note_names(slice_measures(rh, 1, 2)) == ["C", "C", "C", "B"]

# Measure 4 has a sharp — it must read as 'C sharp', not 'C#'
assert to_note_names(slice_measures(rh, 4, 4)) == ["C sharp", "D"], \
    to_note_names(slice_measures(rh, 4, 4))

# Left hand measure 1 is two C-E-G chords
assert to_note_names(slice_measures(lh, 1, 1)) == ["C E G", "C E G"], \
    to_note_names(slice_measures(lh, 1, 1))

print("✅ Task 5 passed")
print("   right hand m1-2 :", to_note_names(slice_measures(rh, 1, 2)))
print("   right hand m4   :", to_note_names(slice_measures(rh, 4, 4)))
print("   left hand  m1   :", to_note_names(slice_measures(lh, 1, 1)))

✅ Task 5 passed
   right hand m1-2 : ['C', 'C', 'C', 'B']
   right hand m4   : ['C sharp', 'D']
   left hand  m1   : ['C E G', 'C E G']


---

# 🔊 Listen

Chain the five functions together and listen to each of the three "hands" once.

In [ ]:
# Right hand, first two measures
listen(slice_measures(rh, 1, 2))

In [ ]:
# Left hand, first two measures — the chords sound like three notes at once
listen(slice_measures(lh, 1, 2))

In [ ]:
# Both hands together, first two measures
listen(slice_measures(get_hand(score, "both"), 1, 2))

右手比較小聲

In [ ]:
# Right hand, the whole piece
listen(slice_measures(rh, 1, prof["n_measures"]))

NameError: name 'prof' is not defined

---

# Task 6: `render_arpeggio`

Break chords **into single notes** while **keeping the rhythm unchanged**.

```
left hand m1:  [C E G]      [C E G]      <- two chords, one beat each
arpeggio:       C  E  G      C  E  G     <- broken apart, total duration unchanged
```

A chord of k notes with duration d → **k notes of duration d/k**. The passage occupies exactly the same amount of time, so the measures stay aligned.

## Spec

- Return a **new** stream (do not modify the original)
- Break chords low to high; each note's duration = original duration ÷ number of notes
- Copy single notes and rests through unchanged
- Use `insert(offset, ...)` to place everything back at its original point in time

## ⚠️ You must use `flatten()` + `offset`

Only after `flatten()` is each note's `.offset` an **absolute time from the beginning**, which is what makes `out.insert(offset, note)` land in the right place.

Using `append()` strings every note into a single line and destroys rests and simultaneous notes.

## This mode does nothing to the right hand

You measured it in Task 3: `rh_n_chords = 0`. **No chords means nothing to break** — arpeggio on the right hand sounds identical to the plain score. It exists for the left hand's 28 chords.

> Hints:
> - `out = m21.stream.Stream()`
> - `for e in stream.flatten().notesAndRests:`
> - a chord's pitches are `e.pitches`; each note's duration is `e.quarterLength / len(e.pitches)`
> - `out.insert(offset + i * step, m21.note.Note(p, quarterLength=step))`

In [ ]:
def render_arpeggio(stream):
    """Break chords into single notes while preserving the original rhythm.

    A k-note chord of duration d becomes k notes of duration d/k, so total time is unchanged.
    """
    out = m21.stream.Stream()

    # TODO: walk stream.flatten().notesAndRests and record each e's offset
    #       chord       -> step = duration / number of pitches; insert the i-th note at offset + i * step
    #       single note -> insert at offset with its original duration
    #       rest        -> copy it through too (use m21.note.Rest), otherwise the rhythm collapses
    raise NotImplementedError

In [ ]:
# ===== Check 6 =====
# Left hand m1: two C-E-G chords become 6 notes, but the total duration is unchanged
written = slice_measures(lh, 1, 1)
broken  = render_arpeggio(written)
assert [n.nameWithOctave for n in broken.recurse().getElementsByClass(m21.note.Note)] == \
    ["C3", "E3", "G3", "C3", "E3", "G3"]
assert float(broken.highestTime) == float(written.highestTime) == 2.0, \
    f"total duration after breaking should still be 2.0, got {float(broken.highestTime)}"

# Every note is one third of the original
lengths = {float(n.quarterLength) for n in broken.recurse().getElementsByClass(m21.note.Note)}
assert lengths == {1/3}, f"every note should be 1/3 beat, got {lengths}"

# The right hand has no chords — breaking it is a no-op
rh6 = slice_measures(rh, 6, 6)
same = render_arpeggio(rh6)
assert [(n.name, float(n.quarterLength)) for n in same.recurse().getElementsByClass(m21.note.Note)] == \
       [(n.name, float(n.quarterLength)) for n in rh6.recurse().getElementsByClass(m21.note.Note)], \
    "the right hand has no chords, so arpeggio should leave it untouched"

# The original stream must not be modified
before = [n.nameWithOctave for n in written.recurse().getElementsByClass(m21.note.Note)]
render_arpeggio(written)
assert [n.nameWithOctave for n in written.recurse().getElementsByClass(m21.note.Note)] == before, \
    "the original stream was modified"

print("✅ Task 6 passed")
print("   left hand m1 broken :", [n.nameWithOctave for n in broken.recurse().getElementsByClass(m21.note.Note)])
print("   total duration      :", float(broken.highestTime), "(same as the original)")

---

# 🔊 Hear the difference

Confirm with your ears whether arpeggio actually does anything — **you can hear it in the left hand, and not in the right.**

In [34]:
print("(1) As written — left hand measure 1 (a chord: three notes at once)")
listen(slice_measures(lh, 1, 1))

(1) As written — left hand measure 1 (a chord: three notes at once)


In [ ]:
print("(2) arpeggio — same measure, broken apart but the same total duration")
listen(render_arpeggio(slice_measures(lh, 1, 1)))

In [36]:
print("Right hand measure 6: arpeggio has no effect on the right hand (rh_n_chords = 0), so both sound the same")
print("   as written:"); listen(slice_measures(rh, 1, 6))
print("   arpeggio:");   listen(render_arpeggio(slice_measures(rh, 6, 6)))

Right hand measure 6: arpeggio has no effect on the right hand (rh_n_chords = 0), so both sound the same
   as written:


   arpeggio:


NameError: name 'render_arpeggio' is not defined

---

# 🎧 Final task: proofread the OMR

**This is the one thing only you can do.**

This score was scanned from an image by a machine, so **it definitely contains errors — we just don't know where**. Another OMR tool (oemer) read the same image as **28 measures**; Audiveris read it as **24**. At least one of them is wrong.

Known Audiveris misses: the title `Allegretto`, the tempo marking (which is why we hard-code 108), and the key signature (the `<key>` tag does not exist at all). **What else is missing or misread in the notes themselves takes a human ear to find.**

## How to do it

1. Play it measure by measure (the cell below)
2. Compare against the original scanned image
3. When you hear a problem, note "which measure, what it should be, what it sounds like" — you can hand that to an AI later to patch the MusicXML directly

## Skip the logging in this lesson

We are shipping a product and time is tight. **You do not actually need to fill in findings** — listening, comparing, and knowing that OMR makes mistakes is enough.

In [ ]:
# Listen measure by measure and compare against the original scan
for i in range(1, prof["n_measures"] + 1):
    print(f"--- measure {i} ---")
    listen(slice_measures(rh, i, i))

In [ ]:
# SKIP this session — we are building the product and time is tight.
# Do not fill in findings.
#
# Later, if you want another AI to patch the MusicXML, notes like this help:
#   m6 | score has D E F D eighths | sounds like ...

findings = ""
print("skipped: no OMR log this session")

---

# Next lesson

```
[done] score reading, preprocessing, playback modes  ->  state & command parsing (instructor writes, you read)  ->  voice interface
```

Today you built all three playback modes:

| Mode | Meaning | Who built it |
|---|---|---|
| `chord` | as written | nothing to write — it is the original score |
| `arpeggio` | chords broken apart, rhythm unchanged | Task 6 ✅ |
| `read_notes` | do not play, speak the pitch names | `to_note_names` from Task 5 ✅ |

"Play it slower" is not a mode — that is `slower` / `faster` adjusting the tempo, which is independent of whether chords get broken apart.

Next lesson has no coding: you will read the **system brain** the instructor wrote (tracking what is currently playing, generating what to say, extracting a measure range from an English sentence), and then we wire it up to a microphone together.